<a href="https://colab.research.google.com/github/jc13605-0721/ECE_9533_LLM4ChipDesign/blob/main/chipchat_exampleA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting set up

## Setting up the notebook

In [ ]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

## Use LLM to generate Verilog code from natural language description

### Example 1: binary_to_bcd

In [ ]:
! mkdir -p binary_to_bcd
! cd binary_to_bcd && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1078  100  1078    0     0   5198      0 --:--:-- --:--:-- --:--:--  5258


In [ ]:
verilog_generation_prompt = '''
Write Verilog-2001 for a combinational 5-bit binary to 8-bit BCD converter using double-dabble.

Module name: binary_to_bcd_converter
Ports: input [4:0] binary_input, output [7:0] bcd_output (bcd_output[7:4]=tens, [3:0]=ones)

Must use double-dabble with correct order each iteration:
1) if any BCD digit >= 5 add 3
2) shift left and shift in next binary bit from MSB to LSB

Output ONLY the module code.

'''

Now you can use the openai library to generate text from your prompt.

Before using LLMs for Verilog generation, you have to first get the api-key:

openai llms: https://platform.openai.com/api-keys
deepseek llms: https://platform.deepseek.com/sign_in
free llms provided by NVIDIA: https://build.nvidia.com/explore/discover

In [ ]:
from google.colab import userdata
import os
from openai import OpenAI

# Load API key from Colab Secrets into environment
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize OpenAI client (API key is read automatically)
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": verilog_generation_prompt}],
    max_tokens=1024,
)

print(completion.choices[0].message.content)


```verilog
module binary_to_bcd_converter(
    input [4:0] binary_input,
    output reg [7:0] bcd_output
);

    integer i;

    always @* begin
        // Initialize BCD output
        bcd_output = 8'b00000000;

        // Perform double-dabble conversion
        for (i = 4; i >= 0; i = i - 1) begin
            // Step 1: If any BCD digit >= 5, add 3
            if (bcd_output[7:4] >= 5)
                bcd_output[7:4] = bcd_output[7:4] + 3;
            if (bcd_output[3:0] >= 5)
                bcd_output[3:0] = bcd_output[3:0] + 3;

            // Step 2: Shift left and shift in the next binary bit
            bcd_output = {bcd_output[6:0], binary_input[i]};
        end
    end

endmodule
```


Next, extract the generated verilog code from LLM response.

In [ ]:
output_verilog_code = '''
module binary_to_bcd_converter(
    input [4:0] binary_input,
    output reg [7:0] bcd_output
);

    integer i;

    always @* begin
        // Initialize BCD output
        bcd_output = 8'b00000000;

        // Perform double-dabble conversion
        for (i = 4; i >= 0; i = i - 1) begin
            // Step 1: If any BCD digit >= 5, add 3
            if (bcd_output[7:4] >= 5)
                bcd_output[7:4] = bcd_output[7:4] + 3;
            if (bcd_output[3:0] >= 5)
                bcd_output[3:0] = bcd_output[3:0] + 3;

            // Step 2: Shift left and shift in the next binary bit
            bcd_output = {bcd_output[6:0], binary_input[i]};
        end
    end

endmodule
'''
filename = "binary_to_bcd/binary_to_bcd.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

Use iverilog to verify the correctness of LLM generated verilog

In [ ]:
!cd binary_to_bcd/ && iverilog -g2012 -o binary_to_bcd.vvp binary_to_bcd.v binary_to_bcd_tb.v && vvp binary_to_bcd.vvp

Testing Binary-to-BCD Converter...
VCD info: dumpfile my_design.vcd opened for output.
All test cases passed!
